In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch01-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "01-04-mol-and-sdf.py"

# 把三種結構和水溶解度一起交給同事

第一次執行會下載 ESOL。先看第一筆 SDF 文字，再對照圖中的三個實際讀回的結構。要做練習，可把程式加在分析格末尾，重新執行分析格與最後一格。


### 先看一份完整的 Mol 紀錄

下面是 RDKit 由苯酚結構產生的 V2000 Mol 紀錄，保留了原本的空白與換行。座標是為顯示結構而計算的二維座標；下表行號由第一行 Phenol 起算，空白行也算一行。[下載這份 Mol 檔](https://raw.githubusercontent.com/517justin/cheminfo-intro/main/book/data/ch01/01-04-phenol.mol)。

```text
Phenol
     RDKit          2D

  7  7  0  0  0  0  0  0  0  0999 V2000
   -0.3214   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.1786   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.9286    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.1786    1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.3214    1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.0714    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.5714   -0.0000    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
  1  2  1  0
  2  3  2  0
  3  4  1  0
  4  5  2  0
  5  6  1  0
  6  7  1  0
  6  1  2  0
M  END
```

| 對照行號 | 這一段在記什麼？ |
|---|---|
| 1–3 | 標頭：第 1 行是名稱 Phenol，第 2 行標示產生程式與 2D，第 3 行在本例留白。 |
| 4 | 計數列：開頭的 `7  7` 表示後面有 7 筆原子與 7 筆鍵資料。 |
| 5–11 | 原子資料：每行前面三個數字是 x、y、z 座標，後面是元素符號；第 11 行的 O 是第 7 個原子。 |
| 12–18 | 鍵資料：每行開頭依序是兩端的原子編號與鍵型，例如 `6  7  1` 是第 6 個碳與第 7 個氧之間的單鍵。 |
| 19 | `M  END`：結束這個分子的結構區塊。 |

### SDF 如何把多個結構和性質放在一起？

SDF 的每筆紀錄先放一個 Mol 結構區塊，再接這個分子的資料項目，最後用獨占一行的 `$$$$` 結束該筆紀錄。下一個分子的標頭就接在後面。因此，`M  END` 是結構區塊的結尾，`$$$$` 才是整筆 SDF 紀錄的結尾；兩者的作用不同。

下面這個可展開的區域是**完整的兩筆 SDF 紀錄**：第一筆是苯酚，第二筆是苯甲醚。結構由 RDKit 寫出，logS 採用本節 ESOL 範例中的數值，並非由座標計算所得。[下載這份雙分子 SDF](https://raw.githubusercontent.com/517justin/cheminfo-intro/main/book/data/ch01/01-04-two-molecules.sdf)。

## SDF 實際內容：苯酚與苯甲醚

```text
Phenol
     RDKit          2D

  7  7  0  0  0  0  0  0  0  0999 V2000
   -0.3214   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.1786   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.9286    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.1786    1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.3214    1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.0714    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.5714   -0.0000    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
  1  2  1  0
  2  3  2  0
  3  4  1  0
  4  5  2  0
  5  6  1  0
  6  7  1  0
  6  1  2  0
M  END
>  <logS>  (1) 
0.0

>  <Endpoint>  (1) 
log10(S / (mol/L))

$$$$
Anisole
     RDKit          2D

  8  8  0  0  0  0  0  0  0  0999 V2000
   -3.0851    0.4695    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.0677   -0.6327    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
   -0.6045   -0.3028    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.4129   -1.4050    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.8762   -1.0750    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.3221    0.3572    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.3047    1.4594    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.1586    1.1294    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
  1  2  1  0
  2  3  1  0
  3  4  2  0
  4  5  1  0
  5  6  2  0
  6  7  1  0
  7  8  2  0
  8  3  1  0
M  END
>  <logS>  (2) 
-1.85

>  <Endpoint>  (2) 
log10(S / (mol/L))

$$$$
```

先找到第一個 `M  END`，往下便會看到 `>  <logS>  (1)`。尖括號裡的 `logS` 是資料項目的名稱，下一行 `0.0` 是苯酚的數值，再下一個空白行表示這個項目的值已結束。後面的 `Endpoint` 說明數值是以 mol/L 表示溶解度後取以 10 為底的對數。RDKit 在資料標頭後附上的 `(1)` 是紀錄序號，不是 logS 的一部分。

接著遇到 `$$$$`，苯酚這筆紀錄就讀完了。下一行 `Anisole` 開始第二筆，後面有苯甲醚自己的原子、鍵與性質，logS 為 `-1.85`。把兩筆的先後順序交換，只要每筆內容保持完整，性質仍會跟著原來的分子結構。

SDF 的性質名稱可由寫檔者設定，也可以省略性質資料，但每筆仍須保留紀錄分隔符號。檔案本身不會替 `logS` 或其他欄位補上單位和量測條件，所以這些意義要像本例的 `Endpoint` 一樣明確交代；只換成 SDF 並不會讓不完整的實驗資訊自動補齊。



In [ ]:
%%writefile "{script}"
from io import BytesIO, StringIO
from pathlib import Path
import os
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor

# 只用三筆 ESOL 量測，標籤保持原始 log10(mol/L) 尺度。
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
names = {"Oc1ccccc1": "Phenol", "COc1ccccc1": "Anisole", "Cc1ccc(O)cc1": "p-Cresol"}
molecules = []
for i, (smiles, value, weight) in enumerate(zip(data.ids, data.y[:, 0], data.w[:, 0])):
    mol = Chem.MolFromSmiles(smiles)
    key = Chem.MolToSmiles(mol)
    if key in names and weight > 0:
        rdDepictor.Compute2DCoords(mol)
        mol.SetProp("_Name", names[key])
        mol.SetProp("RecordID", f"ESOL-row-{i + 1}")
        mol.SetProp("logS", str(float(value)))
        mol.SetProp("Endpoint", "log10(S / (mol/L))")
        molecules.append(mol)
assert len(molecules) == len({m.GetProp("_Name") for m in molecules}) == 3

def as_sdf(mols):
    buffer = StringIO()
    with Chem.SDWriter(buffer) as writer:
        writer.SetProps(["RecordID", "logS", "Endpoint"])
        for mol in mols:
            writer.write(mol)
    return buffer.getvalue()

def read_sdf(text):
    return list(Chem.ForwardSDMolSupplier(BytesIO(text.encode("utf-8"))))

sdf = as_sdf(molecules)
restored = read_sdf(sdf)
assert len(restored) == len(molecules) and all(m is not None for m in restored)
for original, mol in zip(molecules, restored):
    assert Chem.MolToSmiles(original) == Chem.MolToSmiles(mol)
    for field in ["_Name", "RecordID", "logS", "Endpoint"]:
        assert original.GetProp(field) == mol.GetProp(field)
    assert not mol.GetConformer().Is3D()
    print(mol.GetProp("_Name"), mol.GetProp("RecordID"),
          mol.GetNumAtoms(), mol.GetNumBonds(), mol.GetProp("logS"))
print("record separators:", sdf.splitlines().count("$$$$"))
print(sdf.split("$$$$")[0])
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
legends = [f"{m.GetProp('_Name')}   |   {m.GetProp('RecordID')}\nlog S = {float(m.GetProp('logS')):.2f}"
           for m in restored]
figure = Draw.MolsToGridImage(restored, molsPerRow=3, subImgSize=(320, 230), legends=legends)
figure.save(out / "01-04-mol-and-sdf.png")


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "01-04-mol-and-sdf.png")))